# Synthetic Panel Generators — Experimental Bench

**Simulation data generation only.** This notebook extracts the data-generating
processes used by the experiment suite, with every entity name fictional and all
comments in English. No estimator, guard, experiment or reporting code is included.

Two generators live here, both producing the **same column schema** as the baseline
weekly `SKU x region x week` master panel:

| Generator | Purpose |
|---|---|
| `make_lab_panel` | A **single-presentation, single-regime** panel with explicit knobs (number of price moves, move magnitude, true elasticity, pass-through, structural break, competitor activity and reaction). Used to sweep one design dimension at a time. |
| `make_extended_panel` | A **richer portfolio panel** (up to 30 regions, 8 presentations) that fixes, by construction, six limitations of the baseline generator: more regions for the donor pool, heterogeneous and state-dependent pass-through, a regional income component, injected up/down elasticity asymmetry, an actively moving and reacting competitor, and liquidity-driven order refusal. |

Both are accompanied by their **ground-truth tables**, which state what an estimator
run on the panel *should* recover.

### The estimand: why the truth is `beta * rho`, not `beta`

Volume is generated against the **shelf** price, while the price lever (and the price
a demand model regresses on) is the **list** price:

$$\log Q = c + \beta \log\!\big(P^{\text{shelf}}_{\text{real}} / \bar P^{\text{shelf}}\big) + \gamma \log P^{\text{comp}} + \dots$$

$$P^{\text{shelf}}_{\text{real}} = \bar P^{\text{shelf}} \cdot (P^{\text{list}}_{\text{real}}/p_0)^{\rho} \cdot (1 - 0.6\,\text{promo depth})$$

so that

$$\log Q = c + \underbrace{\beta\rho}_{\theta} \log\!\big(P^{\text{list}}_{\text{real}}/p_0\big) + \dots$$

**Consequence.** The elasticity with respect to the list price is $\theta=\beta\rho$.
$\beta$ is the *consumer* elasticity (against the shelf price) and is only recovered
after deconvolving the pass-through $\rho$. In the `sticky_shelf` regime the shelf
price is pinned to a psychological round number, the list price does not enter volume
at all, $\rho = 0$ and therefore $\theta = 0$: that presentation must be **rejected**
by the diagnostics, not estimated.

All figures are in generic currency units; brands, categories and regions are invented.


In [1]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

np.set_printoptions(suppress=True)
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 40)

SEED    = 7      # fixed seed -> full reproducibility
N_WEEKS = 120    # length of the weekly panel

# ---- shared calendar / geography constants ----
REGIONS  = ['North', 'Coastal', 'Central', 'Midlands', 'East', 'South']
REG_TEMP = {'North': 22, 'Coastal': 30, 'Central': 14, 'Midlands': 21, 'East': 26, 'South': 19}

# public holidays per calendar month, used to build the working-days series
HOLIDAYS_BY_MONTH = {1: 1, 2: 0, 3: 1, 4: 1, 5: 1, 6: 1, 7: 0, 8: 1, 9: 0, 10: 0, 11: 1, 12: 2}

print(f"Config -> SEED={SEED} | weeks={N_WEEKS} | base regions={len(REGIONS)}")


Config -> SEED=7 | weeks=120 | base regions=6


## 1 · Ground truth injected in the baseline generator

Reference table for the six-presentation baseline panel: the consumer elasticity
`beta`, the pass-through `rho`, and the resulting list-price estimand
`theta_sellin = beta * rho` per regime. Only the `clean` regime is identifiable by
construction; the other four exist precisely so the diagnostics have something to
reject.


In [2]:
PT_TRUE    = 0.85   # injected pass-through: share of a list-price move reaching the shelf
CROSS_TRUE = 0.30   # cross term on log(competitor price)
FWD_TRUE   = 0.30   # sell-in inflation in the 2 weeks before an increase above 2%

BETA_CONSUMER = {'clean': -1.3, 'sticky_shelf': -1.1, 'weak': -1.0,
                 'confounded': -1.4, 'collinear': -1.2}

# effective pass-through by regime (0 when the shelf price is pinned to a round number)
RHO_BY_REGIME = {'clean': PT_TRUE, 'sticky_shelf': 0.0, 'weak': PT_TRUE,
                 'confounded': PT_TRUE, 'collinear': PT_TRUE}

THETA_SELLIN_TRUE = {k: BETA_CONSUMER[k] * RHO_BY_REGIME[k] for k in BETA_CONSUMER}

PRES_REGIME = {
    'CSD2500_Aurora':  'clean',
    'CSD1500_Aurora':  'clean',
    'CSD1250_Aurora':  'weak',
    'CSD600_Nordia':   'sticky_shelf',
    'WTR600_Vertis':   'confounded',
    'WTR1500_Vertis':  'collinear',
}
PRES_BRAND = {
    'CSD2500_Aurora': 'Aurora', 'CSD1500_Aurora': 'Aurora',
    'CSD1250_Aurora': 'Aurora', 'CSD600_Nordia':  'Nordia',
    'WTR600_Vertis':  'Vertis', 'WTR1500_Vertis': 'Vertis',
}


def truth_by_presentation():
    """Injected ground-truth table, one row per presentation."""
    rows = []
    for pid, reg in PRES_REGIME.items():
        rows.append(dict(pres_id=pid, brand=PRES_BRAND[pid], regime=reg,
                         beta_consumer=BETA_CONSUMER[reg],
                         rho=RHO_BY_REGIME[reg],
                         theta_sellin=THETA_SELLIN_TRUE[reg],
                         identifiable=(reg == 'clean')))
    return pd.DataFrame(rows).set_index('pres_id')


truth_by_presentation()


,brand,regime,beta_consumer,rho,theta_sellin,identifiable
pres_id,,,,,,
CSD2500_Aurora,Aurora,clean,-1.3,0.85,-1.105,True
CSD1500_Aurora,Aurora,clean,-1.3,0.85,-1.105,True
CSD1250_Aurora,Aurora,weak,-1.0,0.85,-0.850,False
CSD600_Nordia,Nordia,sticky_shelf,-1.1,0.00,-0.000,False
WTR600_Vertis,Vertis,confounded,-1.4,0.85,-1.190,False
WTR1500_Vertis,Vertis,collinear,-1.2,0.85,-1.020,False


## 2 · `make_lab_panel` — parameterized single-regime bench

The baseline generator hard-codes the number of price moves, their magnitude and the
true elasticity. Several experiments need to *sweep* exactly those quantities, so this
generator exposes them as arguments and produces a single `clean` presentation across
regions.

Two knobs deserve a note:

- **`comp_moves`** — the baseline DGP gives the competitor only a random walk of 0.4%
  weekly standard deviation, which over 120 weeks leaves a log standard deviation of
  ~1.5%. Against demand noise of 0.05 the cross channel sits **below the noise floor**
  and the cross-elasticity is not identifiable by construction. With `comp_moves > 0`
  the competitor genuinely moves and the competitive bracket becomes testable.
- **`comp_react`** — the competitor's reaction to our own price (the true `r`). With
  `comp_react = 0` the competitor is exogenous and `r_hat ~ 0`, so the identity
  `theta_total - theta_cond = theta_cross * r` holds trivially and tests nothing.

`beta_break = (week, beta_post)` injects a structural break: the elasticity switches
from `beta` to `beta_post` at that week.


In [3]:
def make_lab_panel(rng, W=N_WEEKS, n_moves=10, mag=(0.030, 0.075), beta=-1.3, pt=0.85,
                   cross=0.30, beta_break=None, regions=None, p0=5200,
                   comp_moves=0, comp_mag=(0.04, 0.09), comp_react=0.0,
                   pid='Lab_Pres', brand='LabBrand', category='LabCategory'):
    """Single-presentation `clean` panel with explicit knobs.

    beta_break : None or (week, beta_post). Injects a structural break: the
                 elasticity switches from `beta` to `beta_post` at that week.
    comp_moves : number of DISCRETE competitor price moves (0 = random walk only).
    comp_react : competitor reaction to our own price, with a one-week lag.
    """
    regions = list(regions or REGIONS)
    weeks   = pd.date_range('2023-01-02', periods=W, freq='W-MON')
    woy     = weeks.isocalendar().week.values.astype(int)
    month   = weeks.month.values
    infl    = 100 * (1.0016 ** np.arange(W)) * (1 + rng.normal(0, 0.002, W))
    wage    = 100 * (1.0015 ** np.arange(W))
    holid   = np.array([HOLIDAYS_BY_MONTH[m] for m in month])
    bdays   = np.array([22 - HOLIDAYS_BY_MONTH[m] for m in month])

    # variable cost: level plus a few persistent input shocks
    cost0  = 0.50 * p0
    cshock = np.zeros(W)
    for k in rng.choice(np.arange(10, W - 5), size=4, replace=False):
        cshock[k:] += rng.uniform(0.02, 0.06) * p0 * rng.choice([-1, 1])
    cost_real = np.clip(cost0 + cshock, 0.30 * p0, 0.62 * p0)

    # time profile of beta (constant unless a break is injected)
    beta_t = np.full(W, float(beta))
    if beta_break is not None:
        wk_break, beta_post = beta_break
        beta_t[int(wk_break):] = float(beta_post)

    out = []
    for reg in regions:
        # ---- competitor price
        comp = 1.05 * p0 + np.cumsum(rng.normal(0, 0.004 * p0, W))
        if comp_moves > 0:
            cstep = np.zeros(W)
            for k in rng.choice(np.arange(6, W - 4), size=int(comp_moves), replace=False):
                cstep[k:] += rng.uniform(comp_mag[0], comp_mag[1]) * p0 * rng.choice([-1, 1])
            comp = comp + cstep
        comp = np.clip(comp, 0.70 * p0, 1.45 * p0)

        # ---- promotional calendar
        promo = np.zeros(W, int)
        for k in rng.choice(np.arange(4, W - 3), size=rng.integers(6, 11), replace=False):
            promo[k:k + rng.integers(2, 4)] = 1
        promo = (promo > 0).astype(int)
        pint = promo * rng.uniform(0.10, 0.25)

        # ---- list price: discrete steps off promotional weeks
        base = np.full(W, p0, float)
        step = np.zeros(W)
        nm = int(n_moves)
        if nm > 0:
            cand = [k for k in rng.choice(np.arange(6, W - 4), size=nm, replace=False)
                    if promo[k] == 0]
            for k in cand:
                step[k:] += rng.uniform(mag[0], mag[1]) * p0 * rng.choice([-1, 1])
        list_real = np.clip(base + step, 0.72 * p0, 1.55 * p0)

        if comp_react:
            # the rival follows our own price with a one-week lag
            dlp_own = np.diff(np.log(np.clip(list_real, 1, None)), prepend=np.nan)
            dlp_own = np.nan_to_num(dlp_own)
            react   = np.cumsum(comp_react * np.roll(dlp_own, 1))
            comp    = np.clip(comp * np.exp(react), 0.70 * p0, 1.45 * p0)

        # ---- seasonality via regional temperature
        temp = REG_TEMP[reg] + 6 * np.sin(2 * np.pi * woy / 52 - 1.2) + rng.normal(0, 1.0, W)
        seas = 0.05 * (temp - REG_TEMP[reg])

        # ---- shelf price and demand
        sellout_real = (1.12 * p0) * (list_real / p0) ** pt * (1 - 0.6 * pint)
        sellout      = sellout_real * (infl / 100)
        lv = (np.log(1000) + beta_t * np.log(sellout_real / (1.12 * p0))
              + cross * np.log(comp / (1.05 * p0)) + 0.40 * promo + seas
              + 0.10 * np.log(infl / 100) + rng.normal(0, 0.05, W))
        vol_so = np.exp(lv)

        # ---- forward buying ahead of announced increases above 2%
        dlist = np.diff(np.log(np.clip(list_real, 1, None)), prepend=np.nan)
        fwd   = np.zeros(W)
        for k in np.where(dlist > 0.02)[0]:
            if k - 2 >= 0:
                fwd[k - 2:k] += 0.30
        vol_si = np.clip(vol_so * (1 + fwd) + rng.normal(0, 0.02, W) * vol_so, 1, None)

        out.append(pd.DataFrame({
            'sku_id':            f'{pid}_A',
            'pres_id':           pid,
            'brand':             brand,
            'category':          category,
            'region_id':         reg,
            'date':              weeks,
            'vol_sellin':        vol_si,
            'vol_sellout':       vol_so,
            'price_sellin':      list_real * (infl / 100),
            'price_sellout':     sellout,
            'price_competitor':  comp * (infl / 100),
            'cost_var':          cost_real * (infl / 100),
            'promo_flag':        promo,
            'promo_intensity':   pint,
            'temperature':       temp,
            'inflation_index':   infl,
            'wage_index':        wage,
            'holidays_in_month': holid,
            'business_days':     bdays,
            'channel':           'TRADITIONAL',
            'producer_margin':   (list_real - cost_real) / list_real,
        }))
    return pd.concat(out, ignore_index=True)


def theta_true_lab(beta, pt=0.85):
    """True elasticity of volume with respect to the LIST price."""
    return beta * pt


In [4]:
lab = make_lab_panel(np.random.default_rng(SEED), comp_moves=8, comp_react=0.5)

print(f"Lab panel: {lab.shape} | regions={lab.region_id.nunique()} | weeks={lab.date.nunique()}")
print(f"True theta (list price) = beta * pt = {theta_true_lab(-1.3):.4f}")
lab.head(3)


Lab panel: (720, 21) | regions=6 | weeks=120
True theta (list price) = beta * pt = -1.1050


,sku_id,pres_id,brand,category,region_id,date,vol_sellin,vol_sellout,price_sellin,price_sellout,price_competitor,cost_var,promo_flag,promo_intensity,temperature,inflation_index,wage_index,holidays_in_month,business_days,channel,producer_margin
0,Lab_Pres_A,Lab_Pres,LabBrand,LabCategory,North,2023-01-02,775.892085,782.532930,5200.012794,5824.014329,5442.427052,2600.006397,0,0.0,16.131212,100.000246,100.000000,1,21,TRADITIONAL,0.5
1,Lab_Pres_A,Lab_Pres,LabBrand,LabCategory,North,2023-01-09,706.688709,712.008892,5211.431925,5836.803756,5470.617156,2605.715962,0,0.0,16.455574,100.219845,100.150000,1,21,TRADITIONAL,0.5
2,Lab_Pres_A,Lab_Pres,LabBrand,LabCategory,North,2023-01-16,897.149989,915.021072,5213.793148,5839.448325,5475.826822,2606.896574,0,0.0,19.144838,100.265253,100.300225,1,21,TRADITIONAL,0.5


## 3 · `make_extended_panel` — portfolio generator

Fixes, by construction, six limitations of the baseline generator — all of them CPU
hours rather than a quarter of data negotiation:

1. **30 regions instead of 6.** The floor of a permutation p-value drops from
   1/C(6,3) to 1/C(30,8), and the synthetic-control donor pool becomes real.
2. **Pass-through heterogeneous by region *and* state-dependent** on the retailer's
   margin: exercises the Markov chain and the variance attribution.
3. **Regional income with its own variation**: the cash-pressure index stops being a
   single national series, so the screening test can actually run.
4. **Injected asymmetry `eps_up != eps_dn`**: the Dual Beta estimator has something to
   recover instead of measuring its own noise.
5. **Competitor with discrete moves and a reaction function**: cross-elasticity rises
   above the noise floor.
6. **Liquidity-driven order refusal**: an observable extensive margin, so a two-part
   model has signal to recover.

`seed_noise` is the key knob for coverage work: passing it draws demand (and sell-in)
noise from a **separate** RNG. Fixing `rng` and varying `seed_noise` resamples the noise
while **holding the design fixed** — identical prices, promotions, competitor and
temperature. That is the only way to measure coverage **conditional on the design**,
which is the property a within-panel bootstrap claims to have; dispersion measured
across seeds instead mixes in the variation of the realized design.


In [5]:
EXT_REGIONS_30 = [f'R{i:02d}' for i in range(1, 31)]

EXT_PORTFOLIO = [
    # pres_id,        brand,   category,        beta,  mean_rho, regime
    ('EXT_A1_Alpha',  'Alpha', 'Carbonated',    -1.30, 0.85, 'clean'),
    ('EXT_A2_Alpha',  'Alpha', 'Carbonated',    -1.15, 0.80, 'clean'),
    ('EXT_A3_Alpha',  'Alpha', 'Carbonated',    -1.00, 0.85, 'weak'),
    ('EXT_B1_Beta',   'Beta',  'Carbonated',    -1.10, 0.00, 'sticky_shelf'),
    ('EXT_C1_Gamma',  'Gamma', 'Bottled Water', -1.40, 0.85, 'confounded'),
    ('EXT_C2_Gamma',  'Gamma', 'Bottled Water', -1.20, 0.85, 'collinear'),
    ('EXT_C3_Gamma',  'Gamma', 'Bottled Water', -1.25, 0.60, 'clean'),
    ('EXT_D1_Delta',  'Delta', 'Bottled Water', -0.90, 0.85, 'clean'),
]

EXT_REGIME_CFG = {
    'clean':        dict(n_moves=10, mag=(0.030, 0.075), promo_conf=0.0, comp_follow=0.0),
    'weak':         dict(n_moves=2,  mag=(0.004, 0.009), promo_conf=0.0, comp_follow=0.0),
    'sticky_shelf': dict(n_moves=10, mag=(0.030, 0.075), promo_conf=0.0, comp_follow=0.0),
    'confounded':   dict(n_moves=10, mag=(0.030, 0.075), promo_conf=0.9, comp_follow=0.0),
    'collinear':    dict(n_moves=10, mag=(0.030, 0.075), promo_conf=0.0, comp_follow=0.95),
}


def make_extended_panel(rng, W=N_WEEKS, n_reg=30, portfolio=None, p0=5200,
                        rho_sd=0.12, rho_state=0.35, asym=0.0,
                        comp_moves=8, comp_mag=(0.04, 0.09), comp_react=0.5,
                        income_reg_sd=0.020, refusal=0.0, cross=0.30, beta_break=None,
                        seed_noise=None, sigma_demand=0.05):
    """Extended synthetic panel. Same columns as `make_lab_panel`, plus truth columns.

    rho_sd        : dispersion of rho across regions (0 = homogeneous).
    rho_state     : state dependence — how much rho rises when the retailer's margin
                    falls below its target band.
    asym          : injected asymmetry. eps_up = beta*(1-asym), eps_dn = beta*(1+asym).
                    With 0 the Dual Beta must return symmetry; with 0.25 it must detect
                    a gap of 0.5*|beta|.
    income_reg_sd : sd of the REGIONAL component of real income. With 0 the cash-pressure
                    index is national and its t statistic is not finite.
    refusal       : intensity of liquidity-driven order refusal (extensive margin).
    seed_noise    : if given, demand and sell-in noise come from a SEPARATE RNG, so the
                    noise can be resampled while the design is held fixed.
    """
    regions = EXT_REGIONS_30[:int(n_reg)]
    rngn    = np.random.default_rng(seed_noise) if seed_noise is not None else rng
    port    = list(portfolio if portfolio is not None else EXT_PORTFOLIO)

    weeks   = pd.date_range('2023-01-02', periods=W, freq='W-MON')
    woy     = weeks.isocalendar().week.values.astype(int)
    month   = weeks.month.values
    day_m   = weeks.day.values.astype(float)
    infl    = 100 * (1.0016 ** np.arange(W)) * (1 + rng.normal(0, 0.002, W))
    holid   = np.array([HOLIDAYS_BY_MONTH.get(m, 1) for m in month])
    bdays   = np.array([22 - HOLIDAYS_BY_MONTH.get(m, 1) for m in month])

    # income: national component plus a region-specific one
    wage_nat = 100 * (1.0015 ** np.arange(W))
    wage_reg = {r: wage_nat * np.exp(np.cumsum(rng.normal(0, income_reg_sd, W)))
                for r in regions}

    # True cash pressure: pay-cycle calendar plus regional real-income erosion.
    # Income is standardized POOLED (mean and sd over all regions at once), not
    # within region: standardizing per region would erase exactly the regional
    # level variation this generator exists to inject, and the index would collapse
    # back to a national series.
    cp_cal = -np.minimum(np.abs(day_m - 15), np.abs(day_m - 30))
    cp_cal = (cp_cal - cp_cal.mean()) / max(cp_cal.std(), 1e-9)
    _w_all = np.concatenate([np.log(wage_reg[r]) for r in regions])
    _w_mu, _w_sd = float(_w_all.mean()), float(max(_w_all.std(), 1e-9))

    # rho by region: heterogeneous, truncated to [0, 1]
    rho_mult = {r: float(np.clip(1.0 + rng.normal(0, rho_sd), 0.25, 1.6)) for r in regions}
    temp_reg = {r: float(rng.uniform(13, 31)) for r in regions}

    out = []
    for (pid, brand, cat, beta, rho0, regime) in port:
        cfg    = dict(EXT_REGIME_CFG.get(regime, EXT_REGIME_CFG['clean']))
        beta_t = np.full(W, float(beta))
        if beta_break is not None:
            wk, bpost = beta_break
            beta_t[int(wk):] = float(bpost)
        eps_up, eps_dn = beta * (1 - asym), beta * (1 + asym)

        cost0  = 0.50 * p0
        cshock = np.zeros(W)
        for k in rng.choice(np.arange(10, W - 5), size=4, replace=False):
            cshock[k:] += rng.uniform(0.02, 0.06) * p0 * rng.choice([-1, 1])
        cost_real = np.clip(cost0 + cshock, 0.30 * p0, 0.62 * p0)

        for reg in regions:
            # ---- competitor
            comp = 1.05 * p0 + np.cumsum(rng.normal(0, 0.004 * p0, W))
            if comp_moves > 0:
                cstep = np.zeros(W)
                for k in rng.choice(np.arange(6, W - 4), size=int(comp_moves), replace=False):
                    cstep[k:] += rng.uniform(*comp_mag) * p0 * rng.choice([-1, 1])
                comp = comp + cstep

            # ---- promotions
            promo = np.zeros(W, int)
            for k in rng.choice(np.arange(4, W - 3), size=rng.integers(6, 11), replace=False):
                promo[k:k + rng.integers(2, 4)] = 1
            promo = (promo > 0).astype(int)
            pint = promo * rng.uniform(0.10, 0.25)

            # ---- list price
            step = np.zeros(W)
            nm = int(cfg['n_moves'])
            if nm > 0:
                cand = rng.choice(np.arange(6, W - 4), size=nm, replace=False)
                for k in cand:
                    # confounding with promotions: the move snaps onto a promo week
                    if cfg['promo_conf'] > 0 and rng.random() < cfg['promo_conf']:
                        pk = np.where(promo > 0)[0]
                        if len(pk):
                            k = int(rng.choice(pk))
                    elif promo[k] == 1 and cfg['promo_conf'] == 0:
                        continue
                    step[k:] += rng.uniform(*cfg['mag']) * p0 * rng.choice([-1, 1])
            list_real = np.clip(p0 + step, 0.72 * p0, 1.55 * p0)

            # collinearity with the competitor
            if cfg['comp_follow'] > 0:
                lr = np.log(np.clip(comp / (1.05 * p0), 1e-6, None))
                list_real = np.clip(p0 * np.exp(cfg['comp_follow'] * lr),
                                    0.72 * p0, 1.55 * p0)
            # competitor reaction to our own price
            if comp_react:
                dlp = np.nan_to_num(np.diff(np.log(np.clip(list_real, 1, None)), prepend=np.nan))
                comp = comp * np.exp(np.cumsum(comp_react * np.roll(dlp, 1)))
            comp = np.clip(comp, 0.70 * p0, 1.45 * p0)

            # ---- pass-through: heterogeneous and dependent on the margin state
            rho_r = 0.0 if regime == 'sticky_shelf' else float(
                np.clip(rho0 * rho_mult[reg], 0.0, 1.0))
            marg_target = 0.12
            so   = np.empty(W)
            marg = np.empty(W)
            so_prev = 1.12 * p0
            for k in range(W):
                m_prev = (so_prev - list_real[k]) / max(so_prev, 1e-6)
                # if the margin falls below target, the retailer passes through MORE
                rho_k = np.clip(rho_r * (1.0 + rho_state * (marg_target - m_prev) / marg_target),
                                0.0, 1.2) if rho_r > 0 else 0.0
                so[k] = (1.12 * p0) * (list_real[k] / p0) ** rho_k * (1 - 0.6 * pint[k])
                if rho_r == 0:
                    so[k] = (1.12 * p0) * (1 - 0.6 * pint[k])   # pinned to the round number
                marg[k] = (so[k] - list_real[k]) / max(so[k], 1e-6)
                so_prev = so[k]

            # ---- REALIZED pass-through. With `rho_state` > 0 transmission depends on
            # the margin, so rho_r is no longer the estimand: the estimand is the
            # log-log slope actually realized in the series. It is computed and stored
            # here so the truth an estimator is scored against is the truth of THIS
            # panel and not the one in the config dict. Comparing against the config
            # would report generator dispersion as estimator bias.
            _lso_c = np.log(np.clip(so / ((1.12 * p0) * (1 - 0.6 * pint)), 1e-9, None))
            _llist = np.log(np.clip(list_real / p0, 1e-9, None))
            _vv    = float(np.var(_llist))
            rho_eff = (float(np.cov(_lso_c, _llist)[0, 1] / _vv)
                       if _vv > 1e-12 else float(rho_r))

            # ---- demand, with Houck asymmetry on the shelf price
            lso = np.log(np.clip(so / (1.12 * p0), 1e-9, None))
            if abs(asym) > 1e-12:
                d  = np.diff(lso, prepend=lso[0])
                up = np.cumsum(np.clip(d, 0, None))
                dn = np.cumsum(np.clip(d, None, 0))
                effect = eps_up * up + eps_dn * dn
            else:
                effect = beta_t * lso

            temp  = temp_reg[reg] + 6 * np.sin(2 * np.pi * woy / 52 - 1.2) + rng.normal(0, 1.0, W)
            seas  = 0.05 * (temp - temp_reg[reg])
            wag_r = wage_reg[reg]
            cp_true = cp_cal - (np.log(wag_r) - _w_mu) / _w_sd

            lv = (np.log(1000) + effect + cross * np.log(comp / (1.05 * p0))
                  + 0.40 * promo + seas + 0.10 * np.log(infl / 100)
                  + 0.05 * cp_true + rngn.normal(0, sigma_demand, W))
            vol_so = np.exp(lv)

            # ---- sell-in: forward buying plus liquidity-driven order refusal
            dlist = np.nan_to_num(np.diff(np.log(np.clip(list_real, 1, None)), prepend=np.nan))
            fwd = np.zeros(W)
            for k in np.where(dlist > 0.02)[0]:
                if k - 2 >= 0:
                    fwd[k - 2:k] += 0.30
            vol_si = np.clip(vol_so * (1 + fwd) + rngn.normal(0, 0.02, W) * vol_so, 1, None)
            if refusal > 0:
                # probability of NOT ordering rises as cash gets tight
                pz = 1.0 / (1.0 + np.exp(-(refusal * (-cp_true) - 1.6)))
                vol_si = np.where(rngn.random(W) < pz, 0.0, vol_si)

            out.append(pd.DataFrame({
                'sku_id':            f'{pid}_A',
                'pres_id':           pid,
                'brand':             brand,
                'category':          cat,
                'region_id':         reg,
                'date':              weeks,
                'vol_sellin':        vol_si,
                'vol_sellout':       vol_so,
                'price_sellin':      list_real * (infl / 100),
                'price_sellout':     so * (infl / 100),
                'price_competitor':  comp * (infl / 100),
                'cost_var':          cost_real * (infl / 100),
                'promo_flag':        promo,
                'promo_intensity':   pint,
                'temperature':       temp,
                'inflation_index':   infl,
                'wage_index':        wag_r,
                'holidays_in_month': holid,
                'business_days':     bdays,
                'channel':           'TRADITIONAL',
                'producer_margin':   (list_real - cost_real) / list_real,
                # ---- ground-truth columns (never fed to any estimator) ----
                'retailer_margin':   marg,
                'rho_base':          rho_r,
                'rho_realized':      rho_eff,
                'beta_true':         float(beta),
                'eps_up_true':       float(eps_up),
                'eps_dn_true':       float(eps_dn),
                'theta_realized':    float(beta) * rho_eff,
                'cash_pressure_true': cp_true,
                'regime_true':       regime,
            }))
    return pd.concat(out, ignore_index=True)


### 3·b · Ground truth of the extended generator

Two readings of the truth, and the difference matters:

- **`truth_from_panel`** reads `rho` **from the generated panel**. With heterogeneous,
  state-dependent pass-through, `rho` is no longer the number in the config dict — the
  estimand is the log-log slope actually realized in the series. This is the table an
  estimator should be scored against.
- **`truth_from_config`** reports the *intended* truth from the portfolio definition.
  Useful for documentation, but scoring against it would report generator dispersion as
  estimator bias.


In [6]:
def truth_from_panel(panel, asym=0.0):
    """Truth READ from the generated panel, not from the config dict."""
    rows = []
    for pid, gg in panel.groupby('pres_id'):
        rho  = float(gg.rho_realized.mean())
        beta = float(gg.beta_true.iloc[0])
        reg  = str(gg.regime_true.iloc[0])
        eu   = float(gg.eps_up_true.iloc[0])
        ed   = float(gg.eps_dn_true.iloc[0])
        rows.append(dict(
            pres_id=pid, brand=str(gg.brand.iloc[0]), regime=reg,
            beta_consumer=beta,
            rho=rho, rho_sd_regional=float(gg.groupby('region_id').rho_realized.first().std()),
            theta_sellin=beta * rho,
            theta_up=eu * rho, theta_dn=ed * rho,
            theta_asymmetry=abs(eu * rho - ed * rho),
            identifiable=(reg == 'clean')))
    return pd.DataFrame(rows).set_index('pres_id')


def truth_from_config(portfolio=None, asym=0.0, rho_sd=0.0):
    """Intended truth of the extended generator, one row per presentation.

    theta = beta * rho is the estimand against the list price. With rho_sd > 0 the
    effective rho varies by region, so per-presentation theta is an average and is
    flagged as such.
    """
    port = list(portfolio if portfolio is not None else EXT_PORTFOLIO)
    rows = []
    for (pid, brand, cat, beta, rho0, regime) in port:
        rho = 0.0 if regime == 'sticky_shelf' else rho0
        eu, ed = beta * (1 - asym), beta * (1 + asym)
        rows.append(dict(pres_id=pid, brand=brand, category=cat, regime=regime,
                         beta_consumer=beta, rho=rho, theta_sellin=beta * rho,
                         eps_up=eu, eps_dn=ed,
                         # what a Dual Beta on the LIST price should see
                         theta_up=eu * rho, theta_dn=ed * rho,
                         theta_asymmetry=abs(eu * rho - ed * rho),
                         rho_heterogeneous=bool(rho_sd > 0),
                         identifiable=(regime == 'clean')))
    return pd.DataFrame(rows).set_index('pres_id')


In [7]:
ext = make_extended_panel(np.random.default_rng(SEED), asym=0.25, refusal=0.6)

print(f"Extended panel: {ext.shape} | presentations={ext.pres_id.nunique()} "
      f"| regions={ext.region_id.nunique()} | weeks={ext.date.nunique()}")

truth_from_panel(ext).round(4)


Extended panel: (28800, 30) | presentations=8 | regions=30 | weeks=120


,brand,regime,beta_consumer,rho,rho_sd_regional,theta_sellin,theta_up,theta_dn,theta_asymmetry,identifiable
pres_id,,,,,,,,,,
EXT_A1_Alpha,Alpha,clean,-1.30,0.8574,0.1641,-1.1146,-0.8359,-1.3932,0.5573,True
EXT_A2_Alpha,Alpha,clean,-1.15,0.8249,0.1479,-0.9487,-0.7115,-1.1858,0.4743,True
EXT_A3_Alpha,Alpha,weak,-1.00,0.8798,0.1217,-0.8798,-0.6598,-1.0997,0.4399,False
EXT_B1_Beta,Beta,sticky_shelf,-1.10,0.0000,0.0000,-0.0000,-0.0000,-0.0000,0.0000,False
EXT_C1_Gamma,Gamma,confounded,-1.40,0.8587,0.1833,-1.2021,-0.9016,-1.5027,0.6011,False
EXT_C2_Gamma,Gamma,collinear,-1.20,0.8892,0.1365,-1.0670,-0.8003,-1.3338,0.5335,False
EXT_C3_Gamma,Gamma,clean,-1.25,0.6397,0.1305,-0.7996,-0.5997,-0.9995,0.3998,True
EXT_D1_Delta,Delta,clean,-0.90,0.8539,0.1435,-0.7685,-0.5764,-0.9606,0.3842,True


In [8]:
ext.head(3)


,sku_id,pres_id,brand,category,region_id,date,vol_sellin,vol_sellout,price_sellin,price_sellout,price_competitor,cost_var,promo_flag,promo_intensity,temperature,inflation_index,wage_index,holidays_in_month,business_days,channel,producer_margin,retailer_margin,rho_base,rho_realized,beta_true,eps_up_true,eps_dn_true,theta_realized,cash_pressure_true,regime_true
0,EXT_A1_Alpha_A,EXT_A1_Alpha,Alpha,Carbonated,R01,2023-01-02,0.000000,620.635922,5200.012794,5824.014329,5425.154681,2600.006397,0,0.0,20.779103,100.000246,100.178773,1,21,TRADITIONAL,0.5,0.107143,0.843628,0.928389,-1.3,-0.975,-1.625,-1.206906,-1.626204,clean
1,EXT_A1_Alpha_A,EXT_A1_Alpha,Alpha,Carbonated,R01,2023-01-09,0.000000,815.478471,5211.431925,5836.803756,5450.047952,2605.715962,0,0.0,21.764329,100.219845,99.150076,1,21,TRADITIONAL,0.5,0.107143,0.843628,0.928389,-1.3,-0.975,-1.625,-1.206906,0.258265,clean
2,EXT_A1_Alpha_A,EXT_A1_Alpha,Alpha,Carbonated,R01,2023-01-16,976.380791,966.171707,5213.793148,5839.448325,5437.890802,2606.896574,0,0.0,22.895808,100.265253,99.063524,1,21,TRADITIONAL,0.5,0.107143,0.843628,0.928389,-1.3,-0.975,-1.625,-1.206906,1.566077,clean
